<a href="https://colab.research.google.com/github/mohamedmorsy-ai/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedmorsy-ai/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*


**Selected Lane:** Refresh / Content Opportunity Scoring

**Machine Learning Task:** Binary Classification and Ranking

My project focuses on identifying content pages that may require SEO review or optimization.

The primary machine learning task is binary classification, using historical performance trends to distinguish declining pages from non-declining pages.

A model can estimate a decline-related score for each page, which can then support a ranked review queue.

**Input Features:**
- Search impressions
- Organic clicks
- Organic sessions
- Content age
- Average search position
- Click-through rate (CTR)
- Content freshness
- Engagement indicators

**Expected Output:**

A ranked list of content pages with model scores, supporting evidence, and recommended review priorities.

**Business Objective:**

Help SEO specialists and content managers identify pages that deserve further investigation and allocate their limited resources more effectively.

The model is a decision-support tool, not an automatic content editing system.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*


**Target Variable:** is_declining_label

**Target Type:** Binary Classification

For this starter project, the target is defined using the observed historical trend direction:

- 1 = The page has a declining performance trend.
- 0 = The page does not have a declining performance trend.

**Label Definition:**

is_declining_label = 1 if trend_direction == "down", otherwise 0.

**Source of the Label:**

The label is derived from the anonymized starter dataset provided by FlyRank.

**Why This Target?**

Declining performance is a useful preliminary signal for investigating potential content refresh opportunities.

A model trained with this proxy can help demonstrate how classification scores could be used for ranking pages.

**Important Limitations:**

This is a historical proxy label, not a future outcome or a verified refresh opportunity.

It does not prove that a content update will improve performance.

The trend_direction column must not be included as an input feature because it directly defines the target.

**Future Improvement:**

A stronger production model would use historical features available at a decision date to predict decline or recovery over a separate future period, with proper leakage prevention.


## 3. Success metric

*One metric you can defend. What number means 'good'?*


**Primary Evaluation Metric:** Precision@50

Precision@50 measures how many of the 50 highest-ranked pages actually have the positive evaluation label.

**Formula:**

Precision@50 = Number of positive-label pages among the top 50 / 50

**Why This Metric?**

SEO teams often have limited time to review content.

Instead of reviewing thousands of pages, they need a prioritized shortlist of pages supported by evidence.

Precision@50 directly evaluates how accurately the model ranks positive-label pages near the top of that shortlist.

**Success Criteria:**

The trained model should outperform a transparent rule-based baseline when evaluated on the same held-out data.

An initial target is Precision@50 >= 0.60, subject to comparison with the baseline and the label distribution.

This is a proposed target, not a claimed experimental result.

**Secondary Metrics:**

- Average Precision
- ROC-AUC
- Recall
- Classification precision
- Performance across different client groups

**Validation Strategy:**

Use client-level holdout validation so pages from held-out clients are not included in training.

When future-outcome labels become available, use strict time-based validation and leakage checks.

**Business Interpretation:**

A higher Precision@50 means the model places more historically declining pages among the first 50 review candidates.

It does not establish that these pages will benefit from a refresh.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:

import pandas as pd
from IPython.display import display

# ML-03: Frame Your Lane as an ML Task
# Lane: Refresh / Content Opportunity Scoring

# 1. Load the FlyRank starter dataset
url = (
    "https://raw.githubusercontent.com/"
    "mohamedmorsy-ai/flyrank-ml-internship/"
    "main/data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(url)

# 2. Check required columns
required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "content_age_days",
    "trend_direction"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# 3. Prepare numeric columns
df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"], errors="coerce"
)

df["content_age_days"] = pd.to_numeric(
    df["content_age_days"], errors="coerce"
)

# 4. Apply eligibility filters
df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

# One row represents one content page
df = df.drop_duplicates(
    subset="content_id"
).reset_index(drop=True)

# 5. Define the historical proxy target
df["is_declining_label"] = (
    df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

# 6. Select relevant columns for display
feature_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "avg_position",
    "ctr"
]

available_features = [
    col for col in feature_columns
    if col in df.columns
]

display_columns = [
    "content_id",
    "client_id"
] + available_features + [
    "is_declining_label"
]

unit_df = df[display_columns].copy()

# 7. Display the dataframe
print("ML-03: Machine Learning Task Framing")
print("=" * 50)

print("\nUnit of analysis: One content page")
print("Task type: Binary classification and ranking")
print("Target: is_declining_label")

print("\nDataset overview:")
print(f"Total eligible pages: {len(unit_df):,}")
print(f"Number of columns: {len(unit_df.columns)}")

positive = unit_df["is_declining_label"].sum()
negative = len(unit_df) - positive

print(f"Declining pages (1): {positive:,}")
print(f"Non-declining pages (0): {negative:,}")

print(
    f"Declining percentage: "
    f"{unit_df['is_declining_label'].mean() * 100:.2f}%"
)

print("\nSample content pages:")
display(unit_df.head(10))

print("\nTarget distribution:")
display(
    unit_df["is_declining_label"]
    .value_counts()
    .rename_axis("Target")
    .reset_index(name="Number of Pages")
)

# 8. Basic validation
assert unit_df["content_id"].is_unique
assert unit_df["is_declining_label"].isin([0, 1]).all()

print("\nValidation passed.")
print("Each row represents one unique content page.")
print("The binary historical proxy target is defined.")
print("No model has been trained in this notebook.")


ML-03: Machine Learning Task Framing

Unit of analysis: One content page
Task type: Binary classification and ranking
Target: is_declining_label

Dataset overview:
Total eligible pages: 30,000
Number of columns: 9
Declining pages (1): 16,262
Non-declining pages (0): 13,738
Declining percentage: 54.21%

Sample content pages:


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,content_age_days,avg_position,ctr,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,17,187,10.6,0.76,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,445,20.3,0.05,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,141,36.5,0.09,1
3,content_331d6c4de07b,client_19581e27de,11751,58,78,463,6.2,0.49,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,263,44.0,0.13,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,5,147,8.5,0.03,1
6,content_9a34b442b552,client_8722616204,20,0,1,90,7.0,0.00,1
7,content_a63219c6e95a,client_19581e27de,1724,1,28,445,21.2,0.06,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,68,90,46.0,0.09,1
9,content_c27558df2b0c,client_19581e27de,1240,2,3,257,4.9,0.16,1



Target distribution:


,Target,Number of Pages
0,1,16262
1,0,13738



Validation passed.
Each row represents one unique content page.
The binary historical proxy target is defined.
No model has been trained in this notebook.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


**Why Machine Learning Is Useful for This Problem**

Traditional rule-based approaches rely on predefined conditions to identify pages that may need optimization.

For example, a simple rule might recommend reviewing every page that is more than 180 days old and has at least 500 search impressions.

Although this approach is easy to understand, it may overlook important relationships between multiple variables.

**Limitations of Fixed Rules:**

1. A page can be old but still perform well.
2. A recently updated page may still experience declining traffic.
3. Search impressions alone do not explain content performance.
4. Different content types may require different evaluation criteria.
5. Multiple weak signals may jointly indicate a review opportunity.

**Advantages of Machine Learning:**

Machine learning can examine relationships between search visibility, engagement, content age, and other observed signals.

Models such as Logistic Regression, Decision Trees, and Random Forests can produce classification scores that support more flexible ranking.

A learned model may discover useful combinations of signals that are not captured by a single manually defined threshold.

**Proposed Approach:**

1. Establish a transparent rule-based baseline.
2. Prepare the relevant numerical and categorical features.
3. Train an interpretable classification model.
4. Evaluate it against the baseline using held-out clients.
5. Compare Precision@50 and other relevant metrics.
6. Rank candidate pages and explain the reasons behind each recommendation.

**Responsible Machine Learning:**

A more complex model is not automatically better.

I will prefer the simplest approach that provides a meaningful, reproducible improvement over the baseline.

Because the starter target represents a historical trend, its evaluation measures the ability to identify that proxy, not future performance or causal effects.

**Final Business Action:**

The output will support a ranked content review queue that allows SEO specialists to investigate potentially valuable opportunities before making changes.

Human review remains necessary before recommending content updates, consolidation, or removal.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.